In [1]:
!git clone https://github.com/Nusultan11/imdb-sentiment.git
%cd imdb-sentiment
!pip install -e .

Cloning into 'imdb-sentiment'...
remote: Enumerating objects: 467, done.
remote: Counting objects: 100% (467/467), done.
remote: Compressing objects: 100% (268/268), done.
remote: Total 467 (delta 234), reused 375 (delta 142), pack-reused 0 (from 0)
Receiving objects: 100% (467/467), 494.46 KiB | 12.68 MiB/s, done.
Resolving deltas: 100% (234/234), done.
/content/imdb-sentiment
Obtaining file:///content/imdb-sentiment
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for imdb-sentiment (pyproject.toml) ... done
  Created wheel for imdb-sentiment: filename=imdb_sentiment-0.1.0-0.editable-py3-none-any.whl size=5195 sha256=66eea66ce85e5c99c585605174332dca4452298acdc9dbb97ace8e88f3fd0c8c
  Stored in directory: /tmp/pip-ephem-wheel-cache-o5osj7ho/wheels/3e/08/09/1f8f2dc306b20c0f6d4e5a4ddb080c355c7ce988d70820356f
Suc

In [13]:
import sys
sys.path.append("/content/imdb-sentiment/src")
import shutil

import numpy as np
import torch

from imdb_sentiment.data.dataset import load_imdb_dataset
from imdb_sentiment.data.lstm import build_lstm_dataloader, build_lstm_vocabulary
from imdb_sentiment.models.lstm.model import build_lstm_model
from imdb_sentiment.settings import load_config

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from google.colab import files

In [3]:
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))
else:
    print("running on CPU")

torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [4]:
!python -m imdb_sentiment.cli train --config configs/experiments/lstm_bidirectional_v1.yaml

README.md: 7.81kB [00:00, 14.8MB/s]
plain_text/train-00000-of-00001.parquet: 100% 21.0M/21.0M [00:00<00:00, 32.8MB/s]
plain_text/test-00000-of-00001.parquet: 100% 20.5M/20.5M [00:00<00:00, 49.1MB/s]
plain_text/unsupervised-00000-of-00001.p(…): 100% 42.0M/42.0M [00:00<00:00, 68.2MB/s]
Generating train split: 100% 25000/25000 [00:00<00:00, 120387.60 examples/s]
Generating test split: 100% 25000/25000 [00:00<00:00, 189321.83 examples/s]
Generating unsupervised split: 100% 50000/50000 [00:00<00:00, 162251.46 examples/s]
Training finished.
Accuracy: 0.8418
Precision: 0.9031
Recall: 0.7666
F1: 0.8293
Model saved to: /content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_v1/model.pt
Validation metrics saved to: /content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_v1/val_metrics.json


In [10]:
config = load_config("configs/experiments/lstm_bidirectional_v1.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_imdb_dataset()
train_val_split = dataset["train"].train_test_split(test_size=0.2, seed=config.seed)

train_split = train_val_split["train"]
val_split = train_val_split["test"]

x_train = list(train_split["text"])
y_train = [int(label) for label in train_split["label"]]
x_val = list(val_split["text"])
y_val = [int(label) for label in val_split["label"]]

vocabulary = build_lstm_vocabulary(
    texts=x_train,
    max_size=config.model.vocab_size,
)

val_dataloader = build_lstm_dataloader(
    texts=x_val,
    labels=y_val,
    vocabulary=vocabulary,
    max_length=config.model.max_length,
    batch_size=config.model.batch_size,
    shuffle=False,
    seed=config.seed,
)

model = build_lstm_model(config.model)
checkpoint = torch.load(config.paths.model_output, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("setup ok")
print("val size:", len(x_val))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


setup ok
val size: 5000


In [12]:
all_probs = []
all_targets = []

with torch.no_grad():
    for batch_inputs, batch_labels in val_dataloader:
        batch_inputs = batch_inputs.to(device)
        logits = model(batch_inputs)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.extend(probs.tolist())
        all_targets.extend(batch_labels.numpy().tolist())

all_probs = np.array(all_probs)
all_targets = np.array(all_targets)

best_threshold = None
best_f1 = -1.0
best_metrics = None

for threshold in np.arange(0.10, 0.91, 0.01):
    preds = (all_probs >= threshold).astype(int)

    accuracy = accuracy_score(all_targets, preds)
    precision = precision_score(all_targets, preds)
    recall = recall_score(all_targets, preds)
    f1 = f1_score(all_targets, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = float(threshold)
        best_metrics = {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
        }

print("best_threshold:", best_threshold)
print("best_metrics:", best_metrics)

best_threshold: 0.13999999999999999
best_metrics: {'accuracy': 0.845, 'precision': 0.8262344515642669, 'recall': 0.8747007182761373, 'f1': 0.8497770885830588}


In [14]:
artifact_dir = "/content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_v1"
archive_name = "/content/bidirectional_v1_artifacts"

shutil.make_archive(archive_name, "zip", artifact_dir)
files.download(f"{archive_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>